<a href="https://colab.research.google.com/github/harship123-source/EEG-ExPy/blob/master/Full_Segmentation_Pipeline_with_Metadata_Output_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries with CUDA support
!pip install -q torch

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip install -v --force-reinstall --no-cache-dir llama-cpp-python==0.2.90

In [4]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/colab_packages', exist_ok=True)

# Copy compiled files to Drive
shutil.copytree('/usr/local/lib/python3.13/dist-packages/llama_cpp', '/content/drive/MyDrive/colab_packages/llama_cpp', dirs_exist_ok=True)
shutil.copytree('/usr/local/lib/python3.13/dist-packages/llama_cpp_python-0.2.90.dist-info', '/content/drive/MyDrive/colab_packages/llama_cpp_python-0.2.90.dist-info', dirs_exist_ok=True)

Mounted at /content/drive


'/content/drive/MyDrive/colab_packages/llama_cpp_python-0.2.90.dist-info'

In [5]:
#to use in future session

'''
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Add your Drive package directory to Python's path
sys.path.append('/content/drive/MyDrive/colab_packages')

# 3. Verify it imports correctly
import llama_cpp
print("llama_cpp loaded successfully!")
'''

In [21]:
from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=False     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

Model file exists. Size: 4166.07 MB
Model loaded successfully!


In [9]:
prompt = "What is RAG in the context of large language models?"
print(f"\nSending prompt: {prompt}")

response = llm(prompt, max_tokens=256, temperature=0.1)
print("\nResponse:")
print(response["choices"][0]["text"])


Sending prompt: What is RAG in the context of large language models?



llama_print_timings:        load time =      69.09 ms
llama_print_timings:      sample time =       7.47 ms /   168 runs   (    0.04 ms per token, 22499.00 tokens per second)
llama_print_timings: prompt eval time =      69.00 ms /    13 tokens (    5.31 ms per token,   188.42 tokens per second)
llama_print_timings:        eval time =    4337.33 ms /   167 runs   (   25.97 ms per token,    38.50 tokens per second)
llama_print_timings:       total time =    4508.23 ms /   180 tokens



Response:


RAG, or Recall and Agreement, is a metric used to evaluate the performance of large language models in answering factual questions. It measures the model's ability to recall the correct answer from a given set of candidates and agree with the human-annotated answer.

The RAG metric is calculated as the average of recall and agreement scores. Recall is the percentage of questions for which the model's answer is among the top-k human-annotated answers. Agreement is the percentage of questions for which the model's answer matches the human-annotated answer.

RAG is an important metric for evaluating the accuracy and reliability of large language models, especially in applications where factual correctness is crucial, such as information retrieval, question answering, and knowledge base completion.


In [10]:
def open_source_model(prompt):
    # The 'llm' object is assumed to be globally available from cell tFRT510XNaom
    # It's an instance of llama_cpp.Llama

    messages = [
        {"role": "user", "content": prompt}
    ]

    # Use llm.create_chat_completion for structured JSON output
    chat_completion = llm.create_chat_completion(
        messages=messages,
        response_format={"type": "json_object"}, # Request JSON object output
        max_tokens=50 # Keep max_tokens as set by user
    )

    # Extract the content from the response
    return chat_completion["choices"][0]["message"]["content"]

In [11]:
!pip install -q PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.8 MB/s eta 0:00:00


In [12]:
!pip install -q llama-index llama-index-readers-file llama-index-embeddings-huggingface transformers sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 18.3 MB/s eta 0:00:00


In [ ]:
from llama_index.readers.file import PDFReader

loader = PDFReader()
pages = loader.load_data("/content/pharma-blob-sample.pdf")  # Returns one Document per page

# Print preview
print(f"Loaded {len(pages)} pages")
print(pages[0].text[:300])

In [14]:
documents = []
for i, doc in enumerate(pages):
    doc.metadata = {
        "page_number": i + 1,
        "source_file": "pharma-blob-sample.pdf"
    }
    documents.append(doc)

In [15]:
doc_type_array = [
    "Cover Letter",            # Page 1
    "Certificate of Quality",  # Page 2
    "Certificate of Quality",  # Page 3
    "Packaging Specification", # Page 4
    "Packaging Specification", # Page 5
    "BSE/TSE Declaration",     # Page 6
    "Material Description",    # Page 7
    "Supplier Qualification",  # Page 8
    "Supplier Qualification",  # Page 9
    "Chain of Custody",        # Page 10
]

for doc, doc_type in zip(documents, doc_type_array):
    doc.metadata["doc_type"] = doc_type

In [ ]:
for doc in documents[:2]:
    print(doc.metadata)
    print(doc.text[:150], "\n---\n")

In [ ]:
'''
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Use a local embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Build a vector index enriched with metadata (doc_type, page_number, source_file)
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

print("Metadata-enriched index created with", len(documents), "documents.")
'''

In [30]:
import json
from IPython.display import clear_output

DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification",
             "BSE/TSE Declaration", "Material Description", "Supplier Qualification",
             "Chain of Custody", "Other"]

def classify(page_text, prev_text, prev_type):
    prompt = f"""You classify pages from a pharmaceutical supplier document packet.

Document types: {DOC_TYPES}
Hints:
- "Cover Letter" = a letter addressed to someone starting with 'To Whom It May Concern'.
- A page that continues a table, numbered section, or says "page 2 of N"/"continued"
  has the SAME type as the previous page.

Previous page type: {prev_type or "None (this is the first page)"}
End of previous page:
\"\"\"{prev_text[-600:] if prev_text else ""}\"\"\"

Current page:
\"\"\"{page_text[:3000]}\"\"\"

Return ONLY JSON: {{"doc_type": "<one of the types>", "continues_previous": "<Yes/No>"}}"""
    chat = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=60,
    )
    return json.loads(chat["choices"][0]["message"]["content"])

results = []
prev_text, prev_type = None, None
for doc in documents:
    try:
        out = classify(doc.text, prev_text, prev_type)
    except json.JSONDecodeError:
        out = {"doc_type": "Other", "continues_previous": "No"}
    same_type = out["doc_type"] == prev_type
    is_new = "No" if (same_type and out.get("continues_previous") == "Yes") or \
                     (same_type and prev_type is not None) else "Yes"
    results.append((doc.metadata["page_number"], {"is_new_doc": is_new, "doc_type": out["doc_type"]}))
    prev_text, prev_type = doc.text, out["doc_type"]

clear_output()
correct = 0
for (page, out), truth in zip(results, doc_type_array):
    ok = out["doc_type"] == truth
    correct += ok
    print(f"Page {page}: {out}  {'✓' if ok else '✗ expected ' + truth}")
print(f"\nType accuracy: {correct}/{len(results)}")

Page 1: {'is_new_doc': 'Yes', 'doc_type': 'Cover Letter'}  ✓
Page 2: {'is_new_doc': 'Yes', 'doc_type': 'Certificate of Quality'}  ✓
Page 3: {'is_new_doc': 'No', 'doc_type': 'Certificate of Quality'}  ✓
Page 4: {'is_new_doc': 'Yes', 'doc_type': 'Packaging Specification'}  ✓
Page 5: {'is_new_doc': 'No', 'doc_type': 'Packaging Specification'}  ✓
Page 6: {'is_new_doc': 'Yes', 'doc_type': 'BSE/TSE Declaration'}  ✓
Page 7: {'is_new_doc': 'Yes', 'doc_type': 'Material Description'}  ✓
Page 8: {'is_new_doc': 'Yes', 'doc_type': 'Supplier Qualification'}  ✓
Page 9: {'is_new_doc': 'No', 'doc_type': 'Supplier Qualification'}  ✓
Page 10: {'is_new_doc': 'Yes', 'doc_type': 'Chain of Custody'}  ✓

Type accuracy: 10/10


In [31]:
import json

final_output = []
page_in_doc = -1

for i, (page, out) in enumerate(results):
    is_new = "Yes" if i == 0 else out["is_new_doc"]   # first page always starts a doc
    page_in_doc = 0 if is_new == "Yes" else page_in_doc + 1
    final_output.append({
        "page": i,                     # 0-based page index
        "is_new_doc": is_new,
        "doc_type": out["doc_type"],
        "page_in_doc": page_in_doc,
    })

print("[")
print(",\n".join("  " + json.dumps(row) for row in final_output))
print("]")

# Optional: save to file
with open("/content/segmentation_output.json", "w") as f:
    json.dump(final_output, f, indent=2)

[
  {"page": 0, "is_new_doc": "Yes", "doc_type": "Cover Letter", "page_in_doc": 0},
  {"page": 1, "is_new_doc": "Yes", "doc_type": "Certificate of Quality", "page_in_doc": 0},
  {"page": 2, "is_new_doc": "No", "doc_type": "Certificate of Quality", "page_in_doc": 1},
  {"page": 3, "is_new_doc": "Yes", "doc_type": "Packaging Specification", "page_in_doc": 0},
  {"page": 4, "is_new_doc": "No", "doc_type": "Packaging Specification", "page_in_doc": 1},
  {"page": 5, "is_new_doc": "Yes", "doc_type": "BSE/TSE Declaration", "page_in_doc": 0},
  {"page": 6, "is_new_doc": "Yes", "doc_type": "Material Description", "page_in_doc": 0},
  {"page": 7, "is_new_doc": "Yes", "doc_type": "Supplier Qualification", "page_in_doc": 0},
  {"page": 8, "is_new_doc": "No", "doc_type": "Supplier Qualification", "page_in_doc": 1},
  {"page": 9, "is_new_doc": "Yes", "doc_type": "Chain of Custody", "page_in_doc": 0}
]
